In [0]:
from pyspark.sql import DataFrame

In [0]:

def write_delta_table(
    df : DataFrame,
    schema_name:str,
    folder_name: str,
    table_name:str,
    write_mode: str,
    cluster_keys: list | None = None,
    storage_account: str = "alwahabankingdev001"
)-> None:
    
    full_table_path = f"alwaha_banking_dev_001.{schema_name}.{table_name}"
    checkpoint_path = f"abfss://{schema_name}@{storage_account}.dfs.core.windows.net/deltatables/_checkpoints/v3/{table_name}"

    default_options ={
        "mergeSchema" : "true",
        "tblproperties.delta.autoOptimize.optimizeWrite" : "true",
        "tblproperties.delta.autoOptimize.autoCompact" : "true",
        "tblproperties.delta.enableChildFileCompression" : "true"
    }
    query = (df.writeStream
            .format("delta")
            .outputMode(write_mode)
            .option("checkpointLocation", checkpoint_path)
            .options(**default_options)
            .trigger(availableNow=True)
            .toTable(full_table_path)
            )
    
    query.awaitTermination()

    if cluster_keys and len(cluster_keys) > 0:
        cluster_col_str = ", ".join([str(k) for k in cluster_keys])

        try:
            spark.sql(f"ALTER TABLE {full_table_path} CLUSTER BY ({cluster_col_str})")
        except Exception as e:
            print(f"⚠️ Clustering Note: {str(e)}")